# Verify DetaNet model weights (Model Service)

This notebook shows where the model service loads weights and verifies which files are selected.


In [1]:
import os
import sys
from pathlib import Path

def find_upwards(start: Path, rel: Path):
    cur = start
    while True:
        candidate = cur / rel
        if candidate.exists():
            return candidate
        if cur.parent == cur:
            return None
        cur = cur.parent

APP_MAIN = find_upwards(Path.cwd(), Path('apps/model/app/main.py'))
if APP_MAIN is None:
    raise RuntimeError('Could not locate apps/model/app/main.py from current working directory')
REPO_ROOT = APP_MAIN.parents[3]

default_model_code = REPO_ROOT / 'capsule-3259363' / 'code'
MODEL_CODE_DIR = Path(os.getenv('MODEL_CODE_DIR', str(default_model_code)))
if not MODEL_CODE_DIR.is_dir():
    raise RuntimeError(f'MODEL_CODE_DIR does not exist: {MODEL_CODE_DIR}')

os.environ['MODEL_CODE_DIR'] = str(MODEL_CODE_DIR)
if str(MODEL_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_CODE_DIR))

os.chdir(MODEL_CODE_DIR)
print('MODEL_CODE_DIR:', MODEL_CODE_DIR)


MODEL_CODE_DIR: /Users/rahul/Desktop/proteins-ml-copy/capsule-3259363/code


In [2]:
from pathlib import Path

APP_MAIN = Path(APP_MAIN)
MODEL_LOADER = MODEL_CODE_DIR / 'detanet_model' / 'model_loader.py'
SPECTRA = MODEL_CODE_DIR / 'detanet_model' / 'spectra_simulator.py'

def show_lines(path: Path, contains):
    lines = path.read_text().splitlines()
    for idx, line in enumerate(lines, start=1):
        if any(token in line for token in contains):
            print(f'{path}:{idx}: {line}')

print('--- app/main.py (service model init) ---')
show_lines(APP_MAIN, ['ModelState', 'charge_model', 'nn_vib_analysis', 'nmr_calculator', 'uv_model'])

print('--- detanet_model/model_loader.py (weight loading) ---')
show_lines(MODEL_LOADER, ['torch.load', 'def charge_model', 'def uv_model', 'def Hi_model', 'def Hij_model'])

print('--- detanet_model/spectra_simulator.py (nmr/vib weight usage) ---')
show_lines(SPECTRA, ['nmr_model', 'Hi_model', 'Hij_model', 'dedipole_model', 'depolar_model'])


--- app/main.py (service model init) ---
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:17:     charge_model,
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:19:     nn_vib_analysis,
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:20:     nmr_calculator,
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:22:     uv_model,
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:28: class ModelState:
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:31:         self.charge = charge_model(device=self.device)
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:32:         self.vib = nn_vib_analysis(device=self.device, Linear=False, scale=0.965)
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:33:         self.nmr = nmr_calculator(device=self.device)
/Users/rahul/Desktop/proteins-ml-copy/apps/model/app/main.py:34:         self.uv = uv_model(device=self.device)
/Users/rahul/Desktop/proteins-ml-copy/apps/mo

In [3]:
from detanet_model.model_loader import resolve_weight_paths
from pathlib import Path
import time

paths = resolve_weight_paths()

print('--- resolved weight files (model_loader) ---')
for name in sorted(paths):
    p = Path(paths[name])
    exists = p.exists()
    mtime = time.ctime(p.stat().st_mtime) if exists else 'missing'
    print(f'{name:12s} -> {p} | exists={exists} | mtime={mtime}')


--- resolved weight files (model_loader) ---
Hi           -> trained_param/latest/latest_Hi.pth | exists=True | mtime=Wed Jan 14 08:43:03 2026
Hij          -> trained_param/latest/latest_Hij.pth | exists=True | mtime=Wed Jan 14 08:43:04 2026
borden_os    -> trained_param/qm9spectra/borden_os.pth | exists=True | mtime=Wed Jan 14 08:43:08 2026
dedipole     -> trained_param/latest/latest_dedipole.pth | exists=True | mtime=Wed Jan 14 08:43:04 2026
depolar      -> trained_param/latest/latest_depolar.pth | exists=True | mtime=Wed Jan 14 08:43:03 2026
dipole       -> trained_param/latest/latest_dipole.pth | exists=True | mtime=Wed Jan 14 08:43:04 2026
energy       -> trained_param/qm7x/energy.pth | exists=True | mtime=Wed Jan 14 08:43:05 2026
force        -> trained_param/qm7x/force.pth | exists=True | mtime=Wed Jan 14 08:43:05 2026
hyperpolar   -> trained_param/latest/latest_hyperpolar.pth | exists=True | mtime=Wed Jan 14 08:43:05 2026
npacharge    -> trained_param/latest/latest_npacharge.pt

In [4]:
import torch
from pathlib import Path
from detanet_model import model_loader as ml

if not torch.cuda.is_available():
    _orig_load = ml.torch.load
    def _cpu_load(*args, **kwargs):
        kwargs.setdefault('map_location', torch.device('cpu'))
        return _orig_load(*args, **kwargs)
    ml.torch.load = _cpu_load
    print('torch.load patched with map_location=cpu')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

paths = ml.resolve_weight_paths()
tasks = [name for name, p in paths.items() if Path(p).exists()]
missing = [name for name, p in paths.items() if not Path(p).exists()]
if missing:
    print('skipping missing:', sorted(missing))

try:
    models = ml.load_all_models(device=device, tasks=tasks)
    print('loaded models:', sorted(models.keys()))
except Exception as exc:
    print(f'FAILED load_all_models: {exc}')


torch.load patched with map_location=cpu
device: cpu
loaded models: ['Hi', 'Hij', 'borden_os', 'dedipole', 'depolar', 'dipole', 'energy', 'force', 'hyperpolar', 'npacharge', 'octapole', 'polar', 'quadrupole', 'shield_iso_c', 'shield_iso_h']
